# LLM Zoomcamp Module 1 - Homework Solution

This notebook contains the solutions to the Module 1 homework assignment on Elasticsearch and RAG.

In [ ]:
# Install required packages
# !pip install elasticsearch openai requests tiktoken pandas

In [ ]:
import requests
from elasticsearch import Elasticsearch
import tiktoken
import pandas as pd

## Q1: Running Elastic

Run Elastic Search and get cluster information.

In [ ]:
# Check Elasticsearch connection
response = requests.get('http://localhost:9200')
print(response.json())

In [ ]:
# Get version.build_hash
version_info = response.json()
build_hash = version_info['version']['build_hash']
print(f"Build Hash: {build_hash}")

## Getting the Data

In [ ]:
# Download FAQ documents
docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []
for course in documents_raw:
    course_name = course['course']
    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

print(f"Total documents: {len(documents)}")

## Q2: Indexing the Data

Create index with proper mappings (course as keyword, rest as text).

In [ ]:
# Initialize Elasticsearch client
es = Elasticsearch('http://localhost:9200')

# Define index settings and mappings
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "course": {"type": "keyword"},
            "question": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

# Create index
index_name = 'course-questions'
es.indices.delete(index=index_name, ignore_unavailable=True)
es.indices.create(index=index_name, body=index_settings)
print(f"Index '{index_name}' created successfully")

In [ ]:
# Index documents
for doc in documents:
    es.index(index=index_name, document=doc)

print(f"Indexed {len(documents)} documents")

# Q2 Answer: The function used is 'index'

## Q3: Searching

Search for "How do execute a command on a Kubernetes pod?"

In [ ]:
# Search query
query = {
    "query": {
        "multi_match": {
            "query": "How do execute a command on a Kubernetes pod?",
            "fields": ["question^4", "text"],
            "type": "best_fields"
        }
    }
}

# Execute search
results = es.search(index=index_name, body=query)
hits = results['hits']['hits']

print(f"Top result score: {hits[0]['_score']}")
print(f"\nTop result:")
print(f"Question: {hits[0]['_source']['question']}")
print(f"Score: {hits[0]['_score']}")

## Q4: Filtering

Search for "How do copy a file to a Docker container?" filtered by machine-learning-zoomcamp.

In [ ]:
# Filtered search query
query_filtered = {
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": "How do copy a file to a Docker container?",
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    },
    "size": 3
}

# Execute search
results_filtered = es.search(index=index_name, body=query_filtered)
hits_filtered = results_filtered['hits']['hits']

print(f"Top 3 results for 'How do copy a file to a Docker container?' (ML Zoomcamp only):\n")
for i, hit in enumerate(hits_filtered, 1):
    print(f"{i}. {hit['_source']['question']}")
    print(f"   Score: {hit['_score']}\n")

# Q4 Answer: 3rd question
if len(hits_filtered) >= 3:
    print(f"3rd question: {hits_filtered[2]['_source']['question']}")

## Q5: Building a Prompt

In [ ]:
# Build context from Q4 results
context_template = """
Q: {question}
A: {text}
""".strip()

context_entries = []
for hit in hits_filtered:
    context_entries.append(context_template.format(
        question=hit['_source']['question'],
        text=hit['_source']['text']
    ))

context = '\n\n'.join(context_entries)

# Build prompt
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

question = "How do copy a file to a Docker container?"
prompt = prompt_template.format(question=question, context=context)

print(f"Prompt length: {len(prompt)} characters")

## Q6: Tokens

Calculate number of tokens in the prompt.

In [ ]:
# Tokenize
encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(prompt)

print(f"Number of tokens: {len(tokens)}")

## Summary

### Answers:
- Q1: Build hash from Elasticsearch version
- Q2: `index` function
- Q3: Top score (see output above)
- Q4: 3rd question (see output above)
- Q5: Prompt length in characters
- Q6: Number of tokens

### Next Steps:
1. Verify answers match expected options
2. Push to GitHub
3. Submit via Airtable
4. Complete 3 peer reviews